# sr-diffusion x4 prototype demo

This notebook runs public `jwheo/sr-diffusion` checkpoints from Hugging Face.

Use **Runtime -> Change runtime type -> GPU** before running. The checkpoints are research prototypes under a non-commercial license, not polished production SR models.


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 1. Clone the repo and install lightweight dependencies

Colab already provides PyTorch, so this avoids reinstalling `torch`.

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir('/content')
repo = Path('sr-diffusion')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/BitIntx/sr-diffusion.git'], check=True)
os.chdir(repo)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
print('repo:', Path.cwd())

In [ ]:
%pip -q install pyyaml pillow huggingface_hub
%pip -q install -e . --no-deps

## 2. Select a model and download checkpoints

Choose one model variant below. The notebook downloads only the files needed by that variant.

- `prototype_stage4`: smaller 10k Stage 4 condition-start prototype.
- `photo100k_stage4`: current best sampled photo100k checkpoint.
- `photo100k_v2_stage3`: experimental denoise/sharpening checkpoint trained with stronger `photo_v2` degradation. It can remove heavy artifacts, but may over-correct color/contrast on some images.


In [ ]:
import subprocess

MODEL_VARIANT = 'photo100k_stage4'  # 'prototype_stage4', 'photo100k_stage4', or 'photo100k_v2_stage3'

COMMON_FILES = [
    'LICENSE',
    'CHECKPOINT_LICENSE.md',
    'checkpoints/stage1_autoencoder_best_eval_recon.pt',
]

VARIANTS = {
    'prototype_stage4': {
        'config': 'configs/hf/diffusion_stage4_condition.yaml',
        'files': [
            'checkpoints/stage2_latent_pretrain_best_eval_latent.pt',
            'checkpoints/stage4_condition_b32_best_eval_condition_decoded.pt',
        ],
        'note': '10k Stage 4 condition-start prototype.',
    },
    'photo100k_stage4': {
        'config': 'configs/hf/diffusion_photo100k_stage4_condition.yaml',
        'files': [
            'checkpoints/stage2_photo100k_b64_best_eval_latent.pt',
            'checkpoints/stage4_photo100k_condition_b32_best_eval_condition_decoded.pt',
        ],
        'note': 'Current best sampled photo100k Stage 4 checkpoint.',
    },
    'photo100k_v2_stage3': {
        'config': 'configs/hf/diffusion_photo100k_v2.yaml',
        'files': [
            'checkpoints/stage2_photo100k_v2_b64_best_eval_latent.pt',
            'checkpoints/stage3_photo100k_v2_b32_best_eval_noise.pt',
        ],
        'note': 'Experimental denoise/sharpening Stage 3 checkpoint with photo_v2 degradation.',
    },
}

selected = VARIANTS[MODEL_VARIANT]
cmd = ['python', 'scripts/download_hf_checkpoints.py']
for filename in [*COMMON_FILES, *selected['files']]:
    cmd += ['--file', filename]
print('variant:', MODEL_VARIANT)
print(selected['note'])
print('config:', selected['config'])
subprocess.run(cmd, check=True)
CONFIG_PATH = selected['config']


## 3. Upload an image

For single-tile LR inference, upload a 128x128-ish LR image. For larger LR images, set `USE_TILING = True` in the run cell.

For a controlled smoke test, upload an HR image and set `INPUT_MODE = 'hr'` in the next cell. The script will crop/degrade it first.

In [ ]:
from google.colab import files

uploaded = files.upload()
INPUT_IMAGE = next(iter(uploaded))
print('input:', INPUT_IMAGE)

## 4. Run x4 SR

In [ ]:
import subprocess
from pathlib import Path

INPUT_MODE = 'lr'  # 'lr' for a low-resolution input, 'hr' for controlled HR->LR->SR smoke test
USE_TILING = True  # only used with INPUT_MODE='lr'; start with small LR images such as 256x256
TILE_OVERLAP = 32
TILE_BATCH_SIZE = 1
OUTPUT_DIR = Path(f'outputs/colab_demo_{MODEL_VARIANT}')
STEPS = 32
SEED = 123

input_flag = '--input-lr' if INPUT_MODE == 'lr' else '--input-hr'
cmd = [
    'python', 'infer_diffusion.py',
    '--config', CONFIG_PATH,
    input_flag, INPUT_IMAGE,
    '--output-dir', str(OUTPUT_DIR),
    '--steps', str(STEPS),
    '--seed', str(SEED),
]
if INPUT_MODE == 'lr' and USE_TILING:
    cmd += ['--tile', '--tile-overlap', str(TILE_OVERLAP), '--tile-batch-size', str(TILE_BATCH_SIZE)]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 5. View and download the result

In [ ]:
from IPython.display import display
from PIL import Image
from pathlib import Path

out = Path(f'outputs/colab_demo_{MODEL_VARIANT}')
print('variant:', MODEL_VARIANT)
print('LR input')
display(Image.open(out / 'input_lr.png'))
gt = out / 'gt_hr.png'
if gt.exists():
    print('GT HR')
    display(Image.open(gt))
print('SR output')
display(Image.open(out / 'sr_00.png'))


In [ ]:
from google.colab import files
files.download(str(Path(f'outputs/colab_demo_{MODEL_VARIANT}') / 'sr_00.png'))


## Optional: compare variants

Change `MODEL_VARIANT` in the download cell and rerun sections 2, 4, and 5.

For denoise/sharpening review, compare `photo100k_stage4` against `photo100k_v2_stage3` on the same input. The v2 checkpoint can suppress heavy degradation better, but may introduce color/contrast overshoot or small sampling artifacts.
